In [24]:
import pathlib

import numpy as np
import pandas as pd
import cv2 as cv
import mediapipe as mp

from src.video_transcription.points_normalization import normalize_lists

In [25]:
PATH_TO_VIDEOS = '../../../Videos/ASL_dataset_videos/anita_asl'
videos_path = pathlib.Path(PATH_TO_VIDEOS)

In [26]:
for video in videos_path.iterdir():
    print(video)

../../../Videos/ASL_dataset_videos/anita_asl/F.mkv
../../../Videos/ASL_dataset_videos/anita_asl/C.mkv
../../../Videos/ASL_dataset_videos/anita_asl/Q.mkv
../../../Videos/ASL_dataset_videos/anita_asl/A.mkv
../../../Videos/ASL_dataset_videos/anita_asl/V.mkv
../../../Videos/ASL_dataset_videos/anita_asl/E.mkv
../../../Videos/ASL_dataset_videos/anita_asl/S.mkv
../../../Videos/ASL_dataset_videos/anita_asl/I.mkv
../../../Videos/ASL_dataset_videos/anita_asl/P.mkv
../../../Videos/ASL_dataset_videos/anita_asl/B.mkv
../../../Videos/ASL_dataset_videos/anita_asl/N.mkv
../../../Videos/ASL_dataset_videos/anita_asl/L.mkv
../../../Videos/ASL_dataset_videos/anita_asl/W.mkv
../../../Videos/ASL_dataset_videos/anita_asl/G.mkv
../../../Videos/ASL_dataset_videos/anita_asl/K.mkv
../../../Videos/ASL_dataset_videos/anita_asl/Y.mkv
../../../Videos/ASL_dataset_videos/anita_asl/M.mkv
../../../Videos/ASL_dataset_videos/anita_asl/D.mkv
../../../Videos/ASL_dataset_videos/anita_asl/H.mkv
../../../Videos/ASL_dataset_vid

In [27]:
mpHands = mp.solutions.hands
hands = mpHands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)
mpDraw = mp.solutions.drawing_utils

In [28]:
def draw_circles(xs, ys, radius=10, img_size=480, white=False):
    if white:
        img = np.ones((img_size, img_size, 3), dtype=np.uint8) * 255
    else:
        img = np.zeros((img_size, img_size, 3), dtype=np.uint8)

    for i, (x, y) in enumerate(zip(xs, ys)):
        x = int(x * img_size)
        y = int(y * img_size)
        color_code = i % 4
        match color_code:
            case 0:
                color = (0, 0, 255)
            case 1:
                color = (0, 255, 0)
            case 2:
                color = (255, 0, 0)
            case 3:
                color = (255, 255, 0)
            case _:
                color = (0, 0, 0)
        cv.circle(img, (x, y), radius, color, -1)

    return img


import math


def rotate_points(points):
    origin = points[0]
    rotated_points_x = [origin[0]]
    rotated_points_y = [origin[1]]

    p0, p9 = points[0], points[9]
    dx, dy = p9[0] - p0[0], p9[1] - p0[1]
    alpha = math.atan(dx / dy)

    if alpha > 3.14 / 4:
        alpha -= 3.14 / 2
    elif alpha < -3.14 / 4:
        alpha += 3.14 / 2

    for x, y in points[1:]:
        dx, dy = x - origin[0], y - origin[1]

        new_x = origin[0] + (dx * math.cos(alpha) - dy * math.sin(alpha))
        new_y = origin[1] + (dx * math.sin(alpha) + dy * math.cos(alpha))

        rotated_points_x.append(new_x)
        rotated_points_y.append(new_y)

    return rotated_points_x, rotated_points_y


def normalize_lists(xs, ys, rotate=True):
    if rotate:
        xs, ys = rotate_points(list(zip(xs, ys)))
    min_x = min(xs)
    max_x = max(xs)
    min_y = min(ys)
    max_y = max(ys)

    diff_x = max_x - min_x
    diff_y = max_y - min_y

    scale = max(diff_x, diff_y)

    normalized_xs = [(x - min_x) / scale for x in xs]
    normalized_ys = [(y - min_y) / scale for y in ys]

    return normalized_xs, normalized_ys


In [31]:
CSV_FILE_NAME = 'anita_video_dataset_2.csv'
x_columns = [f"x{i:02}" for i in range(21)]
y_columns = [f"y{i:02}" for i in range(21)]
z_columns = [f"z{i:02}" for i in range(21)]

letters = [chr(i) for i in range(65, 91) if chr(i) not in ['J', 'Z']]
print(letters)
print(len(letters))
# columns = x_columns + y_columns + z_columns + letters
# header = pd.DataFrame(columns=columns)
# header.to_csv(CSV_FILE_NAME, mode='a', index=False)
# header

['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']
24


In [30]:
break_flag = False
current_letter = None
for video in sorted(list(videos_path.iterdir())):
    current_letter = video.name.split('.')[0]
    cnt = 0
    if break_flag:
        break
    cap = cv.VideoCapture(str(video))
    while cap.isOpened():
        success, img = cap.read()
        if not success or cnt > 270:
            break
        imgRGB = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        results = hands.process(imgRGB)

        if results.multi_hand_landmarks:
            cnt += 1
            handLms = results.multi_hand_landmarks[0]
            mpDraw.draw_landmarks(img, handLms, mpHands.HAND_CONNECTIONS)
            xs = [p.x for p in handLms.landmark]
            ys = [p.y for p in handLms.landmark]
            zs = [p.z for p in handLms.landmark]
            list_normalized_xs, list_normalized_ys = normalize_lists(xs, ys, rotate=False)
            df_row = pd.DataFrame([
                list_normalized_xs + list_normalized_ys + zs + [int(current_letter == letter) for letter in letters]])
            df_row.to_csv(CSV_FILE_NAME, mode='a', index=False, header=False)
        print(current_letter, cnt)
        if cv.waitKey(1) & 0xFF == ord('q'):
            break_flag = True
            break
        if cv.waitKey(1) & 0xFF == ord('n'):
            break
    print(current_letter, cnt)
    cap.release()
    cv.destroyAllWindows()

A 1
A 2
A 3
A 4
A 5
A 6
A 7
A 8
A 9
A 10
A 11
A 12
A 13
A 14
A 15
A 16
A 17
A 18
A 19
A 20
A 21
A 22
A 23
A 24
A 25
A 26
A 27
A 28
A 29
A 30
A 31
A 32
A 33
A 34
A 35
A 36
A 37
A 38
A 39
A 40
A 41
A 42
A 43
A 44
A 45
A 46
A 47
A 48
A 49
A 50
A 51
A 52
A 53
A 54
A 55
A 56
A 57
A 58
A 59
A 60
A 61
A 62
A 63
A 64
A 65
A 66
A 67
A 68
A 69
A 70
A 71
A 72
A 73
A 74
A 75
A 76
A 77
A 78
A 79
A 80
A 81
A 82
A 83
A 84
A 85
A 86
A 87
A 88
A 89
A 90
A 91
A 92
A 93
A 94
A 95
A 96
A 97
A 98
A 99
A 100
A 101
A 102
A 103
A 104
A 105
A 106
A 107
A 108
A 109
A 110
A 111
A 112
A 113
A 114
A 115
A 116
A 117
A 118
A 119
A 120
A 121
A 122
A 123
A 124
A 125
A 126
A 127
A 128
A 129
A 130
A 131
A 132
A 133
A 134
A 135
A 136
A 137
A 138
A 139
A 140
A 141
A 142
A 143
A 144
A 145
A 146
A 147
A 148
A 149
A 150
A 151
A 152
A 153
A 154
A 155
A 156
A 157
A 158
A 159
A 160
A 161
A 162
A 163
A 164
A 165
A 166
A 167
A 168
A 169
A 170
A 171
A 172
A 173
A 174
A 175
A 176
A 177
A 178
A 179
A 180
A 181
A 182
A 183
A 184
A 18